# Imports

In [ ]:
import json
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from uavloc.ardupilot.chain import ChainScenario, RunnerConfig, probe_arming, run_chain
from uavloc.ardupilot.chain_analysis import (ekf_error_m, find_fix_log, positions_at, read_fix_log,
                                             summarise_flight)
from uavloc.ardupilot.mission import MissionPlan, mission_items, read_waypoints, write_waypoints
from uavloc.ardupilot.params import vehicle_params, write_param_file
from uavloc.ardupilot.scripts import lua_script_path
from uavloc.ardupilot.sitl import SitlConfig, default_ardupilot_dir, telem2_url
from uavloc.builds import build_dir
from uavloc.data.tiles import ESRI_WORLD_IMAGERY, geo_mosaic
from uavloc.features.onnx_segmenter import OnnxSegmenter
from uavloc.geo import LatLon, east_north_m_array, haversine_m
from uavloc.onboard.package import load_package
from uavloc.onboard.package_writer import PackageRequest, write_package
from uavloc.planning.config import MissionInputs, PlannerConfig
from uavloc.planning.localizability import SelfSimilarity
from uavloc.planning.planner import plan
from uavloc.simulation.camera import CameraConfig
from uavloc.track import REPO_ROOT

# Params

In [ ]:
SLUG = "sitl-v1"
START = (49.905, 23.885)        # west of Birky
END = (49.915, 23.925)          # east of Birky
ALTITUDE_M = 40.0
SPEED_MPS = 5.0
SATELLITE_ONNX = "pocs/lraspp-pi5/models/lraspp-satellite.onnx"
DRONE_ONNX = "pocs/lraspp-pi5/models/lraspp-drone.onnx"
TILE_CACHE = "data/derived/tile-cache"
PLANNER_CACHE = "data/derived/planner-cache"
MISSION_ID = "birky-sitl-v1"
HOME_ALT_M = 300.0              # AMSL of the simulated home
HEADING_DEG = 90.0
SPEEDUP = 8.0
RUNNER_PERIOD_S = 0.5           # simulated seconds between onboard steps
RUNNER_NOISE_M = 6.0            # stub noise, std per axis
RUNNER_SEED = 0
FLIGHT_TIMEOUT_S = 3000.0       # simulated seconds
FLIGHTS = {
    "a-planned-blind": {"no_blind": False, "ekf_action": 1},
    "b-no-blind": {"no_blind": True, "ekf_action": 1},
    "c-planned-blind-failsafe-off": {"no_blind": False, "ekf_action": 0},
}
ARMING_PROBES = [(0.5, True), (0.25, True), (0.2, True), (0.1, True), (0.1, False)]   # (runner period s, fixes on the ground)
ARMING_WAIT_S = 40.0            # simulated seconds an arming probe retries

# Data

In [ ]:
build = build_dir(SLUG, create=True)
figures, work = build / "figures", build / "runs" / "work"
shutil.rmtree(work, ignore_errors=True)
inputs = MissionInputs(LatLon(*START), LatLon(*END), ALTITUDE_M, SPEED_MPS)
config = PlannerConfig()
ardupilot_dir = default_ardupilot_dir()

# Model

In [ ]:
segmenter = OnnxSegmenter(REPO_ROOT / SATELLITE_ONNX)
localizability = SelfSimilarity(config.matcher, config.match_gsd_m, config.blind_s_max_m, config.blind_c_min)

# Plan

In [ ]:
result = plan(inputs, config, ESRI_WORLD_IMAGERY, segmenter, localizability,
              REPO_ROOT / TILE_CACHE, REPO_ROOT / PLANNER_CACHE)
route = min(result.routes, key=lambda candidate: (candidate.blind_share, candidate.length_m))
print(f"{len(result.routes)} routes; chosen weight {route.weight:g}: {route.length_m:.0f} m, blind {route.blind_share:.1%}, "
      f"longest blind {route.longest_blind_m:.1f} m, limit {route.blind_limit_m:.1f} m")

# Package

In [ ]:
satellite = geo_mosaic(ESRI_WORLD_IMAGERY, result.area, config.zoom, REPO_ROOT / TILE_CACHE)

In [ ]:
def write_ardupilot_files(directory) -> dict:
    directory.mkdir(parents=True, exist_ok=True)
    waypoints, params = directory / "mission.waypoints", directory / "uavloc.param"
    write_waypoints(waypoints, mission_items(MissionPlan.from_route(route, inputs)))
    write_param_file(params, vehicle_params())
    return {"mission.waypoints": waypoints, "uavloc.param": params, "vision-gps-switch.lua": lua_script_path()}

In [ ]:
request = PackageRequest(result, route, inputs, config, satellite, REPO_ROOT / DRONE_ONNX, CameraConfig(),
                         write_ardupilot_files(work / "ardupilot-src"), mission_id=MISSION_ID)
archive = write_package(work / "packages", request)
package = load_package(archive)
items = read_waypoints(package.root / "ardupilot" / "mission.waypoints")
print(archive.name, f"{archive.stat().st_size / 1e6:.1f} MB;", len(items), "mission items")

# Scenario

In [ ]:
def scenario(run_name: str, period_s: float, no_blind: bool, ekf_action: int, timeout_s: float) -> ChainScenario:
    sitl = SitlConfig(ardupilot_dir, inputs.start, HOME_ALT_M, HEADING_DEG, SPEEDUP, vehicle_params(),
                      work_dir=work / run_name / "sitl")
    runner = RunnerConfig(archive, telem2_url(), work / run_name / "runner-logs", SPEEDUP, period_s, RUNNER_NOISE_M,
                          RUNNER_SEED, no_blind)
    return ChainScenario(sitl, items, runner, ekf_action, timeout_s)

# Arming

In [ ]:
def probe(index: int, period_s: float, no_blind: bool):
    name = f"arming-{index}"
    return probe_arming(scenario(name, period_s, no_blind, 1, 0.0), work / name / "runner.log", ARMING_WAIT_S)

In [ ]:
probes = [probe(index, *setting) for index, setting in enumerate(tqdm(ARMING_PROBES))]

In [ ]:
arming = pd.DataFrame([{"period_s": p.period_s, "fixes_on_ground": nb, "armed_in_auto": p.armed,
                        "refusal": "" if p.armed else next((t for t in reversed(p.texts) if t.startswith(("Arm:", "PreArm:"))), "")}
                       for p, (_, nb) in zip(probes, ARMING_PROBES)])
arming

# Run

In [ ]:
def fly(name: str):
    return run_chain(scenario(name, RUNNER_PERIOD_S, **FLIGHTS[name], timeout_s=FLIGHT_TIMEOUT_S),
                     work / name / "runner.log")

In [ ]:
flights = {name: fly(name) for name in tqdm(FLIGHTS)}

# Analysis

In [ ]:
fix_logs = {name: read_fix_log(find_fix_log(work / name / "runner-logs")) for name in FLIGHTS}
summaries = {name: summarise_flight(flights[name], fix_logs[name], inputs.start, inputs.end) for name in FLIGHTS}

In [ ]:
def flight_row(name: str) -> dict:
    summary = summaries[name]
    return {"flight": name, "touched_down": summary["completed"], "runner_exit": summary["runner_exit_code"],
            "air_time_s": round(summary["airborne_s"]), "closest_to_end_m": round(summary["closest_to_end_m"], 1),
            "landed_from_start_m": round(summary["landed_offset_from_start_m"], 1),
            "fixes_sent": summary["fixes_by_reason"].get("stub", 0), "fixes_blind": summary["fixes_by_reason"].get("blind", 0),
            "fix_rate_hz": round(summary["fix_rate_hz"], 2),
            "err_max_m": round(summary["active_error_auto_only"]["max_m"], 1),
            "err_mean_m": round(summary["active_error_auto_only"]["mean_m"], 2),
            "modes": " > ".join(mode for _, mode in summary["mode_changes"])}

In [ ]:
outcomes = pd.DataFrame([flight_row(name) for name in FLIGHTS])
outcomes

In [ ]:
def blind_table(name: str) -> pd.DataFrame:
    columns = ["last_fix_s", "gap_s", "length_m", "dropped_fixes", "error_peak_m", "error_at_next_fix_m",
               "position_lost_s", "failsafe_s", "mode_after"]
    return pd.DataFrame(summaries[name]["blind_runs"], columns=columns).round(1)

In [ ]:
blind_tables = {name: blind_table(name) for name in FLIGHTS}
blind_tables["a-planned-blind"]

In [ ]:
blind_tables["c-planned-blind-failsafe-off"]

# Plots

In [ ]:
def to_enu(points) -> np.ndarray:
    east_m, north_m = east_north_m_array(inputs.start, [p.lat for p in points], [p.lon for p in points])
    return np.column_stack([east_m, north_m])

In [ ]:
def draw_plan(ax) -> None:
    ax.plot(*to_enu(package.waypoints).T, "k--", lw=1.2, label="planned out and back")
    for stretch in package.manifest["route"]["blind_stretches"]:
        ax.plot(*to_enu([LatLon(*stretch["start"]), LatLon(*stretch["end"])]).T, color="red", lw=5, alpha=0.5)
    ax.plot(*to_enu([inputs.start]).T, "go", label="start / landing")
    ax.plot(*to_enu([inputs.end]).T, "r^", label="end point")

In [ ]:
def draw_flight(ax, name: str) -> None:
    flight = flights[name]
    dropped = [f.time_s for f in fix_logs[name] if f.reason == "blind"]
    ax.plot(*to_enu([p.truth for p in flight.track]).T, lw=1.4, label="SITL truth")
    ax.plot(*to_enu([p.estimate for p in flight.track]).T, lw=0.8, label="EKF estimate")
    ax.plot(*to_enu(positions_at(flight.track, dropped)).T, "r.", ms=2, label="fix dropped (blind)")
    ax.plot(*to_enu([flight.landed]).T, "kx", ms=10, mew=2, label="touchdown")
    ax.set_title(f"{name}: touched down {summaries[name]['landed_offset_from_start_m']:.0f} m from start")

In [ ]:
fig, axes = plt.subplots(len(FLIGHTS), 1, figsize=(12, 4.6 * len(FLIGHTS)), constrained_layout=True)
for ax, name in zip(axes, FLIGHTS):
    draw_plan(ax)
    draw_flight(ax, name)
    ax.set_aspect("equal")
    ax.set_xlabel("east of start, m")
    ax.set_ylabel("north of start, m")
axes[0].legend(fontsize=8, loc="upper left")
fig.savefig(figures / "track-over-route.png", dpi=120)

In [ ]:
def draw_error(ax, name: str) -> None:
    flight, summary = flights[name], summaries[name]
    origin_s = flight.track[0].sim_time_s
    ax.plot([p.sim_time_s - origin_s for p in flight.track], [ekf_error_m(p) for p in flight.track], lw=0.9)
    for run in summary["blind_runs"]:
        end_s = run["next_fix_s"] if run["next_fix_s"] is not None else flight.track[-1].sim_time_s
        ax.axvspan(run["last_fix_s"] - origin_s, end_s - origin_s, color="red", alpha=0.25)
    ax.axvline(flight.switch_s - origin_s, color="green", ls="--", label="switch to our fixes")
    ax.axvline(flight.gps1_off_s - origin_s, color="purple", ls="--", label="simulated GPS 1 off")
    for time_s, mode in summary["mode_changes"][2:]:
        ax.axvline(time_s - origin_s, color="black", ls=":", label=f"mode {mode}")
    ax.set_title(name)
    ax.set_ylabel("EKF error vs truth, m")

In [ ]:
fig, axes = plt.subplots(len(FLIGHTS), 1, figsize=(12, 3.6 * len(FLIGHTS)), constrained_layout=True)
for ax, name in zip(axes, FLIGHTS):
    draw_error(ax, name)
    ax.set_yscale("symlog", linthresh=10)
    ax.legend(fontsize=8, loc="upper left")
axes[-1].set_xlabel("simulated seconds since arming (red: stretches without fixes)")
fig.savefig(figures / "error-vs-time.png", dpi=120)

# Results

In [ ]:
results = {
    "route": {"weight": route.weight, "length_m": route.length_m, "blind_share": route.blind_share,
              "longest_blind_m": route.longest_blind_m, "blind_limit_m": route.blind_limit_m,
              "exceeds_blind_limit": route.exceeds_blind_limit, "waypoints": len(route.waypoints),
              "blind_stretches": package.manifest["route"]["blind_stretches"]},
    "package_mb": archive.stat().st_size / 1e6,
    "home_offset_m": haversine_m(flights["b-no-blind"].home, inputs.start),
    "mission_differences": {name: flight.mission_differences for name, flight in flights.items()},
    "speedup": SPEEDUP, "runner_period_s": RUNNER_PERIOD_S, "runner_noise_m": RUNNER_NOISE_M,
    "arming": json.loads(arming.to_json(orient="records")),
    "flights": summaries,
}
(build / "results.json").write_text(json.dumps(results, indent=2, default=float))